# Module 41 — Marketing Mix Modeling (MMM) with Regression

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 6 (Statistical Tests)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Marketing Data](#3-synthetic-marketing-data)
4. [Adstock Transformation](#4-adstock-transformation)
5. [MMM Regression](#5-mmm-regression)
6. [Channel Attribution](#6-channel-attribution)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why Marketing Mix Modeling?

MMM attributes **revenue to marketing channels**:
- **Channel effectiveness**: Which channels drive most revenue.
- **Budget optimization**: Allocate spend to highest ROI channels.
- **Saturation effects**: Diminishing returns at high spend.

**Applications**:
1. **Budget allocation**: Optimize spend across channels.
2. **Campaign planning**: Forecast impact of planned campaigns.
3. **Performance tracking**: Monitor channel effectiveness over time.

**Key concepts**:
- **Adstock**: Carryover effect of advertising.
- **Saturation**: Diminishing returns at high spend.
- **Base vs incremental**: Organic vs marketing-driven revenue.

In this notebook we will:
1. Generate synthetic marketing spend and revenue data.
2. Apply adstock transformations.
3. Fit regression model to attribute revenue.
4. Optimize budget allocation.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Marketing Data

In [ ]:
# Generate synthetic marketing data
N_WEEKS = 104  # 2 years

# Marketing channels
channels = ['tv', 'social', 'search', 'email']

# Generate spend data
data = {
    'week': pd.date_range('2022-01-01', periods=N_WEEKS, freq='W'),
}

for channel in channels:
    # Base spend with seasonality
    base = np.random.uniform(50000, 200000)
    seasonal = 20000 * np.sin(np.linspace(0, 4 * np.pi, N_WEEKS))
    noise = np.random.normal(0, 10000, N_WEEKS)
    data[f'{channel}_spend'] = np.maximum(0, base + seasonal + noise)

df = pd.DataFrame(data)

# Generate revenue (correlated with spend + noise)
df['revenue'] = (
    500000  # Base revenue
    + 0.5 * df['tv_spend'] * 0.8  # TV effect
    + 0.3 * df['social_spend'] * 1.2  # Social effect
    + 0.4 * df['search_spend'] * 1.5  # Search effect
    + 0.2 * df['email_spend'] * 2.0  # Email effect
    + np.random.normal(0, 50000, N_WEEKS)  # Noise
)

print(f"✓ Generated {N_WEEKS} weeks of marketing data")
print(f"\nAverage weekly spend:")
for channel in channels:
    print(f"  {channel}: ${df[f'{channel}_spend'].mean():,.0f}")
print(f"\nAverage weekly revenue: ${df['revenue'].mean():,.0f}")

---
## §4 · Adstock Transformation

In [ ]:
# Apply adstock transformation
def adstock(series, decay_rate=0.5):
    """Apply adstock transformation to model carryover effects."""
    adstocked = series.copy()
    for i in range(1, len(series)):
        adstocked.iloc[i] = series.iloc[i] + decay_rate * adstocked.iloc[i-1]
    return adstocked

# Apply to each channel
for channel in channels:
    df[f'{channel}_adstock'] = adstock(df[f'{channel}_spend'], decay_rate=0.5)

print(f"✓ Adstock transformation applied")
print(f"  Decay rate: 0.5")
print(f"\nSample adstock values:")
print(df[['tv_spend', 'tv_adstock']].head())

---
## §5 · MMM Regression

In [ ]:
# Prepare features
feature_cols = [f'{channel}_adstock' for channel in channels]
X = df[feature_cols]
y = df['revenue']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit regression model
model = LinearRegression()
model.fit(X_scaled, y)

# Predictions
y_pred = model.predict(X_scaled)

# Metrics
r2 = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print(f"✓ MMM regression fitted")
print(f"  R²: {r2:.4f}")
print(f"  MAE: ${mae:,.0f}")
print(f"\nChannel coefficients (scaled):")
for channel, coef in zip(channels, model.coef_):
    print(f"  {channel}: {coef:.4f}")

---
## §6 · Channel Attribution

In [ ]:
# Calculate channel attribution
attribution = pd.DataFrame({
    'channel': channels,
    'coefficient': model.coef_,
    'avg_spend': [df[f'{channel}_spend'].mean() for channel in channels],
    'contribution': model.coef_ * [df[f'{channel}_adstock'].mean() for channel in channels]
})

# Calculate ROI
attribution['roi'] = attribution['contribution'] / attribution['avg_spend']

# Sort by contribution
attribution = attribution.sort_values('contribution', ascending=False)

print(f"✓ Channel attribution calculated")
print(f"\nChannel Performance:")
print(attribution.to_string(index=False))

# Optimal budget allocation
total_budget = attribution['avg_spend'].sum()
attribution['optimal_pct'] = attribution['roi'] / attribution['roi'].sum() * 100
attribution['optimal_spend'] = total_budget * attribution['optimal_pct'] / 100

print(f"\nOptimal budget allocation:")
for _, row in attribution.iterrows():
    print(f"  {row['channel']}: ${row['optimal_spend']:,.0f} ({row['optimal_pct']:.1f}%)")

---
## §7 · Visualization

In [ ]:
# Visualize channel performance
fig = make_subplots(rows=1, cols=2, subplot_titles=['Channel Contribution', 'Current vs Optimal Spend'])

# Channel contribution
fig.add_trace(go.Bar(
    x=attribution['channel'],
    y=attribution['contribution'],
    name='Contribution',
    marker_color='#636EFA'
), row=1, col=1)

# Current vs Optimal spend
fig.add_trace(go.Bar(
    x=attribution['channel'],
    y=attribution['avg_spend'],
    name='Current Spend',
    marker_color='#EF553B'
), row=1, col=2)

fig.add_trace(go.Bar(
    x=attribution['channel'],
    y=attribution['optimal_spend'],
    name='Optimal Spend',
    marker_color='#00CC96'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Search has highest ROI")
print("   - Email is most cost-effective")
print("   - Reallocate budget from low to high ROI channels")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Marketing Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Budget planning, channel optimization, performance reporting |
> | **Model** | Regression with adstock transformations |
> | **Features** | Spend by channel, seasonality, external factors |
> | **Evaluation** | R², MAE, contribution analysis |
> | **Deployment** | Monthly model refresh, quarterly budget planning |
>
> ### 💡 Budget Optimization
>
> | Step | Action | Frequency |
> |------|--------|-----------|
> | 1 | Collect spend and revenue data | Weekly |
> | 2 | Fit MMM with adstock | Monthly |
> | 3 | Calculate channel ROI | Monthly |
> | 4 | Optimize budget allocation | Quarterly |
> | 5 | A/B test new allocation | Quarterly |
>
> ### 🔑 Key Takeaways
>
> 1. **MMM attributes revenue to marketing channels** quantitatively.
> 2. **Adstock captures carryover effects** of advertising.
> 3. **ROI varies significantly** across channels.
> 4. **Budget optimization** can improve overall ROI by 20-30%.
> 5. **Regular model updates** capture changing channel effectiveness.